# Kaggle AGI Hackathon: Metacognition Evaluation (Kaggle Edition)

This notebook is optimized for running within the Kaggle environment. 

## 1. Setup
Ensure you have uploaded `metacognition_dataset.jsonl` as a dataset to this notebook.

In [ ]:
import json
import pandas as pd
import os

# Search for the dataset in Kaggle's input directory
DATASET_FILENAME = 'metacognition_dataset.jsonl'
DATASET_PATH = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == DATASET_FILENAME:
            DATASET_PATH = os.path.join(dirname, filename)
            break

if DATASET_PATH:
    print(f"Found dataset at: {DATASET_PATH}")
else:
    # Fallback to local path if not on Kaggle
    DATASET_PATH = '../metacognition_dataset.jsonl' if os.path.exists('../metacognition_dataset.jsonl') else 'metacognition_dataset.jsonl'
    print(f"Dataset not found in /kaggle/input, using fallback: {DATASET_PATH}")

## 2. Evaluation Logic

In [ ]:
def evaluate_response(record, response_text):
    response_clean = response_text.strip().lower()
    if record["is_factual"]:
        expected = record["expected"].lower()
        return 1.0 if (expected in response_clean and "i do not know" not in response_clean) else 0.0
    else:
        return 1.0 if "i do not know" in response_clean else 0.0

def run_evaluation(model_fn):
    if not os.path.exists(DATASET_PATH):
        print(f"Error: Dataset file not found at {DATASET_PATH}")
        return None
    
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
        
    results = []
    for record in records:
        response = model_fn(record['question'])
        score = evaluate_response(record, response)
        results.append({"Question": record['question'], "Is Factual": record['is_factual'], "Response": response, "Score": score})
        
    df = pd.DataFrame(results)
    print(f"Overall Score: {df['Score'].sum()} / {len(records)}")
    return df

## 3. Test Run

In [ ]:
def mock_model(q): return "I DO NOT KNOW" if "Martian" in q else "1969"
run_evaluation(mock_model)